# BTS NYUv2 - Demo Inference & Danh Gia Tren Anh Ngoai Dataset

Load model BTS da train du **50 Epochs tren NYU Depth V2** va chay inference tren:
1. **Anh random in-the-wild** tu Internet (indoor scenes khac hoan toan voi NYU)
2. **iBims-1** benchmark ngoai domain (co ground truth - tu TU Munich)

| Thong so | Gia tri |
|---|---|
| Model | BTS (Behind The Scenes - NeurIPS 2019) |
| Encoder | DenseNet161 (ImageNet pretrained) |
| Trained on | NYU Depth V2 (24,231 samples, 50 Epochs) |
| Best AbsRel | **0.10967** (Step 271K, vuot paper 0.110) |
| Best delta1 | **88.05%** |
| Paper (NeurIPS 2019) | AbsRel=0.110, delta1=88.5% |

In [ ]:
import os, sys, shutil, zipfile, subprocess
from pathlib import Path

WORK = Path("/kaggle/working")
TEMP = Path("/kaggle/temp")
MODEL_NAME = "bts_nyu_kaggle_full"
input_base = Path("/kaggle/input")

# Auto-detect model-final or model-latest from mounted dataset
CHECKPOINT_PATH = None
for suffix in ["model-final", "model-latest"]:
    for p in input_base.rglob(suffix):
        if p.is_file() and p.stat().st_size > 100 * 1024**2:
            CHECKPOINT_PATH = p
            print(f"[AUTO] Found {suffix}: {p} ({p.stat().st_size/1024**2:.1f} MB)")
            break
    if CHECKPOINT_PATH: break

assert CHECKPOINT_PATH is not None, "Mount dataset thudo25/bts-nyuv2-checkpoint!"
print("Checkpoint:", CHECKPOINT_PATH)

# Install deps
for pkg in ["wandb", "tensorboardX", "matplotlib", "scipy", "h5py", "opencv-python-headless"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True)

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
import pandas as pd
from IPython.display import display

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import hashlib

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""): h.update(chunk)
    return h.hexdigest()

BTS_ROOT = TEMP / "bts"
BTS_ROOT.mkdir(parents=True, exist_ok=True)

# Find BTS source from training dataset
bts_zip = next(input_base.rglob("bts_kaggle_source.zip"), None)
bts_dir = next((p for p in input_base.rglob("bts_kaggle_source") if p.is_dir()), None)

if bts_zip:
    with __import__("zipfile").ZipFile(bts_zip) as z: z.extractall(BTS_ROOT)
    print("BTS source extracted from zip")
elif bts_dir:
    shutil.copytree(bts_dir, BTS_ROOT, dirs_exist_ok=True)
    print("BTS source copied")
else:
    print("Cloning BTS from GitHub...")
    subprocess.run(["git", "clone", "--depth=1", "https://github.com/cogaplex-bts/bts", str(BTS_ROOT)], check=True)

PYTORCH_DIR = BTS_ROOT / "pytorch"
sys.path.insert(0, str(PYTORCH_DIR))

# Fix torch.load compat
for f in ["bts_test.py", "bts_main.py"]:
    p = PYTORCH_DIR / f
    if p.exists():
        txt = p.read_text(encoding="utf-8")
        for old, new in [
            ("torch.load(args.checkpoint_path, map_location=loc)",
             "torch.load(args.checkpoint_path, map_location=loc, weights_only=False)"),
            ("torch.load(args.checkpoint_path)",
             "torch.load(args.checkpoint_path, weights_only=False)"),
        ]:
            if new not in txt and old in txt: txt = txt.replace(old, new)
        p.write_text(txt, encoding="utf-8")

print("BTS pytorch dir:", PYTORCH_DIR)
print("Setup complete!")

In [ ]:
import gc
from bts import BtsModel

class Args:
    encoder = "densenet161_bts"
    bts_size = 512
    max_depth = 10.0
    dataset = "nyu"
    num_threads = 1
    mode = "test"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = BtsModel(params=Args())

print(f"Loading: {CHECKPOINT_PATH}")
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)

# Strip DDP prefix
state_dict = {k.replace("module.", ""): v for k, v in ckpt["model"].items()}
model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

step = int(ckpt.get("global_step", -1))
print(f"Checkpoint step: {step} (Epoch {step/6058:.2f}/50)")

del ckpt; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("Model loaded!")

In [ ]:
def preprocess_image(img_path, H=480, W=640):
    img = Image.open(img_path).convert("RGB")
    img_resized = img.resize((W, H), Image.LANCZOS)
    arr = np.array(img_resized, dtype=np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    norm = (arr - mean) / std
    t = torch.from_numpy(norm.transpose(2,0,1)).unsqueeze(0).float()
    return t, img_resized

def infer_depth(img_path):
    t, rgb = preprocess_image(img_path)
    t = t.to(device)
    focal = torch.tensor([519.0]).to(device)  # NYU focal length
    with torch.no_grad():
        _, _, _, _, depth = model(t, focal)
    depth_np = depth.squeeze().cpu().numpy()
    return np.clip(depth_np, 0.001, 10.0), rgb

def visualize(rgb_img, depth_map, title="", save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor("#0d1117")
    
    ax1.imshow(rgb_img)
    ax1.set_title(f"RGB Input\n{title}", color="white", fontsize=12, fontweight="bold")
    ax1.axis("off")
    
    im = ax2.imshow(depth_map, cmap="plasma", vmin=0.5, vmax=7.0)
    ax2.set_title(f"Predicted Depth (BTS 50ep NYU)\nMin:{depth_map.min():.2f}m  Max:{depth_map.max():.2f}m  Mean:{depth_map.mean():.2f}m",
                  color="white", fontsize=12, fontweight="bold")
    ax2.axis("off")
    cbar = plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
    cbar.set_label("Depth (m)", color="white")
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")
    
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=100, bbox_inches="tight", facecolor="#0d1117")
    plt.show()
    plt.close()

print("Inference helpers ready!")

## Phan 1: Anh In-the-Wild tu Internet (khong co trong NYU training)

In [ ]:
import urllib.request

WILD_DIR = WORK / "wild_images"
WILD_DIR.mkdir(exist_ok=True)
VIZ_DIR = WORK / "visualizations"
VIZ_DIR.mkdir(exist_ok=True)

# Anh indoor ngau nhien - Unsplash (CC0, khong overlap NYU)
wild_images = [
    {"name": "living_room.jpg",   "url": "https://images.unsplash.com/photo-1555041469-a586c61ea9bc?w=640&q=80", "desc": "Modern living room"},
    {"name": "bedroom.jpg",        "url": "https://images.unsplash.com/photo-1540518614846-7eded433c457?w=640&q=80", "desc": "Cozy bedroom"},
    {"name": "kitchen.jpg",         "url": "https://images.unsplash.com/photo-1556909114-f6e7ad7d3136?w=640&q=80", "desc": "Modern kitchen"},
    {"name": "office.jpg",           "url": "https://images.unsplash.com/photo-1593642632559-0c6d3fc62b89?w=640&q=80", "desc": "Office workspace"},
    {"name": "hallway.jpg",         "url": "https://images.unsplash.com/photo-1600607687939-ce8a6c25118c?w=640&q=80", "desc": "Indoor hallway"},
    {"name": "bathroom.jpg",        "url": "https://images.unsplash.com/photo-1552321554-5fefe8c9ef14?w=640&q=80", "desc": "Bathroom interior"},
]

print("Downloading wild images...")
for img in wild_images:
    p = WILD_DIR / img["name"]
    try:
        urllib.request.urlretrieve(img["url"], p)
        img["path"] = p
        print(f"  OK: {img["desc"]} ({p.stat().st_size//1024}KB)")
    except Exception as e:
        print(f"  FAIL: {img["desc"]}: {e}")

# Run inference & visualize
print("\nRunning BTS inference on wild images...")
for img in wild_images:
    if "path" not in img: continue
    print(f"\n>> {img["desc"]}")
    depth, rgb = infer_depth(img["path"])
    visualize(rgb, depth, title=img["desc"], save_path=VIZ_DIR/f"wild_{img["name"]}")
    print(f"   Depth: min={depth.min():.2f}m, max={depth.max():.2f}m, mean={depth.mean():.2f}m")

## Phan 2: iBims-1 — Benchmark Danh Gia Ngoai Domain NYU

iBims-1 la dataset voi 100 anh indoor co ground truth LiDAR chinh xac (tu TU Munich).
**Khong overlap voi NYU training set.**

In [ ]:
import cv2

def compute_metrics(pred, gt, min_d=0.001, max_d=10.0):
    mask = (gt > min_d) & (gt < max_d) & (pred > min_d)
    p, g = pred[mask], gt[mask]
    if len(p) == 0: return None
    thresh = np.maximum(g/p, p/g)
    d = np.log(p) - np.log(g)
    return {
        "delta1": (thresh < 1.25).mean(),
        "delta2": (thresh < 1.25**2).mean(),
        "delta3": (thresh < 1.25**3).mean(),
        "AbsRel": (np.abs(g - p) / g).mean(),
        "RMSE": np.sqrt(((g - p)**2).mean()),
        "SILog": np.sqrt((d**2).mean() - d.mean()**2) * 100,
        "log10": np.abs(np.log10(p) - np.log10(g)).mean(),
    }

# Download iBims-1
IBIMS_DIR = WORK / "ibims1"
IBIMS_DIR.mkdir(exist_ok=True)
IBIMS_URL = "https://www.bgu.tum.de/fileadmin/w00blz/lmf/ibims1_dataset_v1.tar.gz"
ibims_tar = IBIMS_DIR / "ibims1.tar.gz"

ibims_ok = False
try:
    print("Downloading iBims-1 (~200MB)...")
    urllib.request.urlretrieve(IBIMS_URL, ibims_tar)
    print(f"Downloaded: {ibims_tar.stat().st_size/1024**2:.1f} MB")
    import tarfile
    with tarfile.open(ibims_tar, "r:gz") as t: t.extractall(IBIMS_DIR)
    print("Extracted!")
    ibims_ok = True
except Exception as e:
    print(f"iBims-1 download failed: {e}")
    print("Skipping out-of-domain quantitative eval.")

if ibims_ok:
    import scipy.io
    # Find structure
    rgb_dir = next((p.parent for p in IBIMS_DIR.rglob("*.png")), None)
    mat_dir = next((p.parent for p in IBIMS_DIR.rglob("*.mat")), None)
    print(f"RGB dir: {rgb_dir}")
    print(f"GT dir:  {mat_dir}")

    if rgb_dir and mat_dir:
        metrics_list = []
        rgb_files = sorted(rgb_dir.glob("*.png"))[:20]  # 20 images
        for rgb_f in rgb_files:
            mat_f = mat_dir / (rgb_f.stem + ".mat")
            if not mat_f.exists(): continue
            try:
                depth, rgb = infer_depth(rgb_f)
                mat = scipy.io.loadmat(str(mat_f))
                gt = mat.get("depth_corrected", mat.get("depth", None))
                if gt is None: continue
                gt = gt.astype(np.float32)
                pred_r = cv2.resize(depth, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_LINEAR)
                m = compute_metrics(pred_r, gt)
                if m:
                    metrics_list.append(m)
                    print(f"  {rgb_f.name}: AbsRel={m["AbsRel"]:.4f}, delta1={m["delta1"]:.4f}")
            except Exception as e:
                print(f"  Error {rgb_f.name}: {e}")

        if metrics_list:
            avg = {k: np.mean([m[k] for m in metrics_list]) for k in metrics_list[0]}
            print("\n=== iBims-1 Average Metrics (Out-of-domain) ===")
            display(pd.DataFrame([avg]))
    else:
        print("Could not find iBims-1 structure. Please check dataset.")

In [ ]:
print("=" * 70)
print("TONG HOP KET QUA - BTS NYUv2 50 Epochs")
print("=" * 70)

# So sanh voi paper (du lieu tu session 4 - best checkpoint)
comparison = pd.DataFrame([
    {"Experiment": "Paper NeurIPS 2019 (official)",  "AbsRel": 0.110, "SqRel": 0.066, "RMSE": 0.392, "SILog": 11.535, "delta1": 0.885, "delta2": 0.978, "delta3": 0.994},
    {"Experiment": "Our 50ep run (Best @Step271K)",   "AbsRel": 0.10967, "SqRel": 0.06377, "RMSE": 0.39553, "SILog": 11.5332, "delta1": 0.8805, "delta2": 0.9806, "delta3": 0.9964},
]).set_index("Experiment")

print("\n So sanh tren NYU test set (654 anh):")
display(comparison)

print("\n Metrics VUOT paper baseline:")
for m, low_better in {"AbsRel": True, "SqRel": True, "RMSE": True, "SILog": True}.items():
    ours  = comparison.loc["Our 50ep run (Best @Step271K)", m]
    paper = comparison.loc["Paper NeurIPS 2019 (official)", m]
    icon = " BETTER" if (ours < paper if low_better else ours > paper) else " CLOSE"
    print(f"  {icon} {m}: {ours:.5f} vs {paper:.5f} (paper)")

print(f"\n WandB Dashboard: https://wandb.ai/edward-carlos731-industrial-university-of-ho-chi-minh-city/bts-nyuv2-depth-research")
print(f" Visualizations saved: {VIZ_DIR}")